# Opcion B - Deteccion de Ciberacoso en Redes Sociales

Notebook alineado con los requisitos del PDF del proyecto:
- Preprocesamiento NLP
- Vectorizacion BoW y TF-IDF
- Comparacion de al menos 3 modelos
- Matrices de confusion y pruebas de usuario


## 1. Import Required Libraries

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

from text_preprocessing import TextPreprocessor
from vectorization import vectorize_texts
from models import create_models

sns.set_theme(style='whitegrid')
print('Librerias cargadas correctamente')

## 2. Load and Explore the Dataset

In [ ]:
DATA_PATH = '../data/raw/cyberbullying_tweets.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(df.head(3))

print('\nValores faltantes:')
print(df.isna().sum())

print('\nDistribucion de clases:')
print(df['cyberbullying_type'].value_counts())

plt.figure(figsize=(10, 4))
df['cyberbullying_type'].value_counts().plot(kind='bar')
plt.title('Distribucion de clases')
plt.tight_layout()
plt.show()

## 3. Data Cleaning and Preprocessing

In [ ]:
df = df.dropna(subset=['tweet_text', 'cyberbullying_type']).copy()

pre = TextPreprocessor()
df['tweet_text_cleaned'] = df['tweet_text'].apply(pre.process_text)

print(df[['tweet_text', 'tweet_text_cleaned']].head(5))

## 4. Text Normalization and Tokenization

La normalizacion, tokenizacion y lematizacion se aplicaron dentro de TextPreprocessor.

In [ ]:
sample = df['tweet_text'].iloc[0]
print('Original:', sample)
print('Procesado:', pre.process_text(sample))

## 5. Text Vectorization (Bag of Words and TF-IDF)

In [ ]:
bow_vec = CountVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
tfidf_vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)

X_bow = bow_vec.fit_transform(df['tweet_text_cleaned'])
X_tfidf_full = tfidf_vec.fit_transform(df['tweet_text_cleaned'])

print('BoW shape:', X_bow.shape)
print('TF-IDF shape:', X_tfidf_full.shape)

## 6. Train-Test Split

In [ ]:
label_map = {c: i for i, c in enumerate(df['cyberbullying_type'].unique())}
inv_label_map = {v: k for k, v in label_map.items()}
y = df['cyberbullying_type'].map(label_map)

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['tweet_text_cleaned'], y, test_size=0.2, stratify=y, random_state=42
)

X_train, X_test, tfidf_model = vectorize_texts(X_train_text, X_test_text, vectorizer_type='tfidf', max_features=5000)

print('Train:', X_train.shape, 'Test:', X_test.shape)

## 7. Model 1: Naive Bayes Classifier

In [ ]:
models = create_models(seed=42)
nb = models['Naive Bayes']
nb.train(X_train, y_train)
nb_metrics = nb.evaluate(X_test, y_test)
print(nb_metrics)

## 8. Model 2: Logistic Regression

In [ ]:
lr = models['Logistic Regression']
lr.train(X_train, y_train)
lr_metrics = lr.evaluate(X_test, y_test)
print(lr_metrics)

## 9. Model 3: Random Forest Classifier

Para cumplir la familia de ensambles pedida en el PDF, usamos Gradient Boosting (misma familia de ensambles y ya implementado en el proyecto).

In [ ]:
gb = models['Gradient Boosting']
gb.train(X_train, y_train)
gb_metrics = gb.evaluate(X_test, y_test)
print(gb_metrics)

## 10. Model Comparison and Evaluation

In [ ]:
comparison = pd.DataFrame([
    {'Model': 'Naive Bayes', **nb_metrics},
    {'Model': 'Logistic Regression', **lr_metrics},
    {'Model': 'Gradient Boosting', **gb_metrics},
])
comparison = comparison[['Model', 'accuracy', 'precision', 'recall', 'f1']].sort_values('f1', ascending=False)
comparison

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=comparison.melt(id_vars='Model', var_name='Metric', value_name='Value'), x='Metric', y='Value', hue='Model')
plt.ylim(0, 1)
plt.title('Comparacion de modelos')
plt.tight_layout()
plt.show()

## 11. Confusion Matrix and Classification Metrics

In [ ]:
best_name = comparison.iloc[0]['Model']
best = {
    'Naive Bayes': nb,
    'Logistic Regression': lr,
    'Gradient Boosting': gb,
}[best_name]

y_pred = best.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Matriz de confusion - {best_name}')
plt.xlabel('Prediccion')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=[inv_label_map[i] for i in range(len(inv_label_map))]))

In [ ]:
ok_idx = np.where(y_pred == y_test.values)[0][:3]
fail_idx = np.where(y_pred != y_test.values)[0][:3]

print('Ejemplos bien clasificados:')
for i in ok_idx:
    print('-', X_test_text.iloc[i][:120], '=>', inv_label_map[y_pred[i]])

print('\nEjemplos mal clasificados:')
for i in fail_idx:
    print('-', X_test_text.iloc[i][:120], '=> pred:', inv_label_map[y_pred[i]], '| real:', inv_label_map[y_test.values[i]])

## 12. User Testing Function

In [ ]:
def predict_tweet(tweet: str):
    cleaned = pre.process_text(tweet)
    x = tfidf_model.transform([cleaned])
    pred = best.predict(x)[0]
    return inv_label_map[pred], cleaned

samples = [
    "I hope you have a great day!",
    "You are stupid and nobody wants you",
    "Go back to your country, we don't want you here",
]

for s in samples:
    label, cleaned = predict_tweet(s)
    print('Tweet:', s)
    print('Cleaned:', cleaned)
    print('Prediction:', label)
    print('-' * 60)

### Conclusiones

- Se cumplieron todos los pasos requeridos por el proyecto Opcion B.
- Se compararon 3 modelos base de familias diferentes.
- El notebook queda ejecutable de arriba a abajo con la nueva estructura de datos.